# Pipeline Maestro — Farmacovigilancia Computacional
## Proyecto Final: Calidad y Preprocesamiento de Datos

**Framework:** TDQM (Total Data Quality Management)
**Fuentes:** SIDER 4.1 (Kuhn et al., 2016) · PubChem REST API (Kim et al., 2023)
**Stack:** pandas · RDKit · recordlinkage · thefuzz · scikit-learn · ydata-profiling · seaborn

---

| # | Fase TDQM | Script original | Contenido |
|---|-----------|----------------|-----------|
| 1 | Adquisición | `descargar_datos.py` | Carga SIDER · descarga PubChem SMILES |
| 2 | Fusión | `fusion.py` | Record linkage · modelo canónico |
| 3 | Medir (antes) | `perfilado.py` | Perfilado línea base · ydata-profiling |
| 4 | Mejorar | `limpieza.py` | Imputación severity · validación RDKit · normalización |
| 5 | Medir (después) | `perfilado.py` | Comparación antes/después |
| 6 | Analizar | `análisis.py` | 20 consultas A–C · fingerprints + Tanimoto + PCA + KMeans |


## 0. Setup — Imports y Rutas

In [ ]:
%matplotlib inline

# ─── Librería estándar ────────────────────────────────────────────────────────
import os, re, time, warnings
import numpy as np
import pandas as pd
from pathlib import Path

# ─── Visualización ───────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ─── Quimioinformática ───────────────────────────────────────────────────────
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors

# ─── Machine Learning ────────────────────────────────────────────────────────
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# ─── Record Linkage / Fuzzy ──────────────────────────────────────────────────
import recordlinkage
from thefuzz import fuzz

# ─── Requests (PubChem API) ──────────────────────────────────────────────────
import requests

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)

# ─── Rutas del proyecto ───────────────────────────────────────────────────────
ROOT     = Path("/Users/EdgarDevice/Desktop/calidadprojecto")
RAW      = ROOT / "Bases_datos" / "raw"
CLEAN    = ROOT / "Bases_datos" / "clean"
PERFILES = CLEAN / "perfiles"
GRAFICAS = CLEAN / "graficas"

for d in [CLEAN, PERFILES, GRAFICAS]:
    d.mkdir(parents=True, exist_ok=True)

# ─── Helper para guardar y mostrar figuras ───────────────────────────────────
def guardar(fig, path, show=True):
    fig.savefig(path, dpi=150, bbox_inches="tight")
    if show:
        plt.show()
    plt.close(fig)

print("✓ Setup completo")
print(f"  RAW      : {RAW}")
print(f"  CLEAN    : {CLEAN}")
print(f"  PERFILES : {PERFILES}")
print(f"  GRAFICAS : {GRAFICAS}")


---
## 1. Adquisición de Datos (`descargar_datos.py`)

### Marco teórico
El proyecto integra dos fuentes heterogéneas:
- **SIDER 4.1** — Base de datos de reacciones adversas a medicamentos. Utiliza la terminología MedDRA (Medical Dictionary for Regulatory Activities). Archivo principal: `meddra_all_se.tsv` con 291,632 registros crudos.
- **PubChem** — Base de datos de estructuras moleculares del NIH. Se consulta via API REST PUG solicitando `IsomericSMILES` e `IUPACName` por lotes de 200 CIDs.

### Clave de unión
El identificador STITCH (formato `CID0XXXXXXXX`) contiene el PubChem CID incrustado. Al quitar el prefijo `CID0` se obtiene el entero que sirve como llave de unión entre SIDER y PubChem.

### Términos MedDRA
Se filtran solo registros de tipo **PT (Preferred Term)** para evitar duplicados jerárquicos (LLT < PT < HLT < HLGT < SOC). El PT es la granularidad clínica estándar.


In [ ]:
print("=" * 60)
print("PASO 1: Cargando SIDER meddra_all_se.tsv")
print("=" * 60)

cols_se = ["stitch_flat", "stitch_stereo", "umls_found",
           "meddra_type", "umls_meddra", "side_effect_name"]

se_raw = pd.read_csv(RAW / "meddra_all_se.tsv",
                     sep="\t", header=None, names=cols_se)

print(f"  Total registros cargados : {len(se_raw):,}")
print(f"  Tipos MedDRA disponibles : {se_raw['meddra_type'].value_counts().to_dict()}")

# Filtrar solo Preferred Terms (PT) — evita duplicados jerárquicos
se_pt = se_raw[se_raw["meddra_type"] == "PT"].copy()
print(f"  Registros PT (filtrados) : {len(se_pt):,}")

# Extraer PubChem CID desde stitch_stereo (CID0XXXXXXXX → int)
se_pt["pubchem_cid"] = (
    se_pt["stitch_stereo"]
    .str.replace("CID0", "", regex=False)
    .astype(int)
)

se_pt = se_pt[["stitch_flat", "pubchem_cid", "umls_meddra", "side_effect_name"]]

# Guardar si no existe ya
out_pt = RAW / "sider_se_pt.csv"
se_pt.to_csv(out_pt, index=False)
print(f"  Guardado: sider_se_pt.csv ({len(se_pt):,} filas)")

se_pt.head(3)


In [ ]:
print("PASO 2: Cargando SIDER drug_names.tsv")

drug_names = pd.read_csv(RAW / "drug_names.tsv",
                         sep="\t", header=None,
                         names=["stitch_flat", "drug_name"])

out_dn = RAW / "sider_drug_names.csv"
drug_names.to_csv(out_dn, index=False)
print(f"  Medicamentos únicos : {drug_names['stitch_flat'].nunique():,}")
print(f"  Guardado: sider_drug_names.csv")

drug_names.head(3)


In [ ]:
# ─── PubChem SMILES ─────────────────────────────────────────────────────────
# Si el archivo ya existe se carga directamente (evita re-descarga ~5 min).
# Para forzar re-descarga: borrar datos/raw/pubchem_smiles.csv y re-ejecutar.

PUBCHEM_OUT = RAW / "pubchem_smiles.csv"

if PUBCHEM_OUT.exists():
    print(f"✓ pubchem_smiles.csv ya existe — cargando...")
    smiles_df = pd.read_csv(PUBCHEM_OUT)
    print(f"  {len(smiles_df):,} registros con SMILES")
else:
    print("Descargando SMILES de PubChem (puede tardar ~5 min)...")

    unique_cids   = sorted(se_pt["pubchem_cid"].unique())
    PUBCHEM_URL   = ("https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid"
                     "/property/IsomericSMILES,IUPACName/JSON")
    BATCH_SIZE    = 200
    results, errors = [], []

    for i in range(0, len(unique_cids), BATCH_SIZE):
        batch     = unique_cids[i : i + BATCH_SIZE]
        batch_num = i // BATCH_SIZE + 1
        total_b   = (len(unique_cids) + BATCH_SIZE - 1) // BATCH_SIZE
        print(f"  Lote {batch_num}/{total_b} — CIDs {batch[0]}..{batch[-1]}", end=" ")
        try:
            resp = requests.post(PUBCHEM_URL,
                                 data={"cid": ",".join(map(str, batch))},
                                 timeout=30)
            if resp.status_code == 200:
                props = resp.json()["PropertyTable"]["Properties"]
                results.extend(props)
                print(f"✓ ({len(props)} registros)")
            else:
                print(f"✗ HTTP {resp.status_code}")
                errors.extend(batch)
        except Exception as e:
            print(f"✗ {e}")
            errors.extend(batch)
        time.sleep(0.3)

    smiles_df = pd.DataFrame(results)
    smiles_df.columns = [c.lower() for c in smiles_df.columns]
    smiles_df = smiles_df.rename(columns={
        "cid": "pubchem_cid",
        "isomericsmiles": "smiles",
        "iupacname": "iupac_name"
    })
    smiles_df.to_csv(PUBCHEM_OUT, index=False)
    print(f"\n  Guardado: pubchem_smiles.csv ({len(smiles_df):,} registros)")
    if errors:
        pd.DataFrame({"pubchem_cid": errors}).to_csv(RAW / "pubchem_errors.csv", index=False)

smiles_df.head(3)


---
## 2. Fusión y Record Linkage (`fusion.py`)

### Marco teórico — Heterogeneidad semántica
La integración de SIDER y PubChem enfrenta **heterogeneidad semántica**: el mismo medicamento tiene nombres radicalmente distintos entre fuentes. Ejemplo:
- SIDER: `"Aspirin"`
- PubChem IUPAC: `"2-acetyloxybenzoic acid"`

### Record Linkage
Se aplica el modelo de **Fellegi-Sunter** a través de la biblioteca `recordlinkage`:
1. **Indexación:** Full index sobre la tabla de CIDs únicos (comparar cada CID consigo mismo en ambas fuentes).
2. **Comparación:** Similitud Jaro-Winkler (robusta a transposiciones) y Levenshtein entre `drug_name` (SIDER) e `iupac_name` (PubChem).
3. **Clasificación:** Umbral τ = 0.85 para alta similitud.

### Estrategia de resolución de conflictos
Preferencia al nombre SIDER (más legible clínicamente). Si no hay nombre SIDER, se usa IUPAC; si ninguno, "desconocido".


In [ ]:
print("=" * 70)
print("FUSION — Construcción del modelo canónico e_drugDB")
print("=" * 70)

# Cargar las tres fuentes raw
se       = pd.read_csv(RAW / "sider_se_pt.csv")
drug     = pd.read_csv(RAW / "sider_drug_names.csv")
smi      = pd.read_csv(RAW / "pubchem_smiles.csv")

print(f"\n[Fuentes cargadas]")
print(f"  sider_se_pt      : {len(se):>8,} filas | "
      f"{se['pubchem_cid'].nunique():,} fármacos | "
      f"{se['side_effect_name'].nunique():,} reacciones")
print(f"  sider_drug_names : {len(drug):>8,} filas | "
      f"{drug['stitch_flat'].nunique():,} fármacos únicos")
print(f"  pubchem_smiles   : {len(smi):>8,} filas | "
      f"{smi['pubchem_cid'].nunique():,} CIDs únicos")


In [ ]:
print("\n[Medición de heterogeneidad entre fuentes]")

cids_se   = set(se["pubchem_cid"].unique())
cids_drug = set(se.merge(drug, on="stitch_flat")["pubchem_cid"].unique())
cids_smi  = set(smi["pubchem_cid"].unique())

print(f"  CIDs en SE sin nombre SIDER  : {len(cids_se - cids_drug):,}  "
      f"({100*len(cids_se - cids_drug)/len(cids_se):.1f}%)")
print(f"  CIDs en SE sin SMILES        : {len(cids_se - cids_smi):,}  "
      f"({100*len(cids_se - cids_smi)/len(cids_se):.1f}%)")

# Muestra de heterogeneidad semántica drug_name vs iupac_name
joined_sample = (
    se[["pubchem_cid", "stitch_flat"]].drop_duplicates()
    .merge(drug, on="stitch_flat", how="inner")
    .merge(smi,  on="pubchem_cid",  how="inner")
    [["pubchem_cid", "drug_name", "iupac_name"]]
    .head(8)
)
print("\n  Muestra de heterogeneidad semántica (drug_name SIDER vs IUPAC PubChem):")
print("  " + "-"*75)
for _, row in joined_sample.iterrows():
    score = fuzz.token_sort_ratio(str(row["drug_name"]).lower(),
                                   str(row["iupac_name"]).lower())
    print(f"  CID {row['pubchem_cid']:>8} | "
          f"SIDER: {str(row['drug_name'])[:22]:<22} | "
          f"IUPAC: {str(row['iupac_name'])[:30]:<30} | sim={score}")


In [ ]:
print("\n[Record Linkage: SIDER drug_name ↔ PubChem iupac_name]")

# Tabla comparativa por CID
cid_names = (
    se[["pubchem_cid", "stitch_flat"]].drop_duplicates()
    .merge(drug, on="stitch_flat", how="left")
    .merge(smi,  on="pubchem_cid",  how="left")
    [["pubchem_cid", "drug_name", "iupac_name"]]
    .drop_duplicates("pubchem_cid")
    .reset_index(drop=True)
)

left  = cid_names[["pubchem_cid", "drug_name"]].copy()
right = cid_names[["pubchem_cid", "iupac_name"]].copy()
left.index  = left["pubchem_cid"]
right.index = right["pubchem_cid"]

indexer = recordlinkage.Index()
indexer.add(recordlinkage.index.Full())
candidate_links = indexer.index(left, right)

compare = recordlinkage.Compare()
compare.string("drug_name", "iupac_name", method="levenshtein", label="sim_lev")
compare.string("drug_name", "iupac_name", method="jarowinkler",  label="sim_jw")
features = compare.compute(candidate_links, left, right)

# Solo pares propios (mismo CID en ambos lados)
own_pairs = features[
    features.index.get_level_values(0) == features.index.get_level_values(1)
].copy()
own_pairs.index = own_pairs.index.get_level_values(0)
own_pairs.index.name = "pubchem_cid"

def categorizar(sim):
    if pd.isna(sim):  return "sin_nombre_sider"
    if sim >= 0.85:   return "alta_similitud"
    if sim >= 0.50:   return "similitud_media"
    return "baja_similitud"

own_pairs["categoria"] = own_pairs["sim_jw"].apply(categorizar)
dist = own_pairs["categoria"].value_counts()
print("  Distribución Jaro-Winkler (drug_name SIDER vs IUPAC PubChem):")
for cat, n in dist.items():
    print(f"    {cat:<25}: {n:>5,}  ({100*n/len(own_pairs):.1f}%)")

# Guardar reporte
linkage_report = cid_names.join(own_pairs[["sim_lev","sim_jw","categoria"]],
                                on="pubchem_cid", how="left")
linkage_report.to_csv(CLEAN / "linkage_report.csv", index=False)
print(f"  Reporte guardado: linkage_report.csv")


In [ ]:
print("\n[Fusión de fuentes y resolución de conflictos]")

merged = (
    se
    .merge(drug, on="stitch_flat", how="left")
    .merge(smi,  on="pubchem_cid",  how="left")
)

def resolver_nombre(row):
    sider = str(row["drug_name"]).strip() if pd.notna(row["drug_name"]) else ""
    iupac = str(row["iupac_name"]).strip() if pd.notna(row["iupac_name"]) else ""
    if sider and sider.lower() not in ("nan", "none", ""):
        return sider
    if iupac and iupac.lower() not in ("nan", "none", ""):
        return iupac
    return "desconocido"

merged["name"]     = merged.apply(resolver_nombre, axis=1)
merged["severity"] = "unknown"
merged["source"]   = merged.apply(
    lambda r: "SIDER+PubChem" if pd.notna(r["smiles"]) else "SIDER_only", axis=1
)
merged["drug_id"]  = "DRUG_" + merged["pubchem_cid"].astype(str).str.zfill(8)

canonical = merged[[
    "drug_id", "name", "smiles", "side_effect_name",
    "umls_meddra", "severity", "source", "pubchem_cid", "stitch_flat"
]].rename(columns={"side_effect_name": "adverse_reaction",
                    "umls_meddra":       "reaction_type"})

canonical["name"]             = canonical["name"].str.strip().str.title()
canonical["adverse_reaction"] = canonical["adverse_reaction"].str.strip().str.title()

# Deduplicar
before = len(canonical)
canonical = canonical.drop_duplicates(subset=["drug_id", "adverse_reaction"])
after  = len(canonical)
print(f"  Antes de deduplicar   : {before:>8,} filas")
print(f"  Duplicados eliminados : {before-after:>8,}  ({100*(before-after)/before:.1f}%)")
print(f"  Modelo canónico       : {after:>8,} filas")

# Subset con SMILES
con_smiles   = canonical[canonical["smiles"].notna()]
sin_smiles_n = canonical["smiles"].isna().sum()
print(f"  Sin SMILES (excluidos): {sin_smiles_n:>8,}")
print(f"  Con SMILES (finales)  : {len(con_smiles):>8,}")

# Guardar
canonical.to_csv(CLEAN / "e_drugDB.csv",       index=False)
con_smiles.to_csv(CLEAN / "e_drugDB_smiles.csv", index=False)
print("\n  e_drugDB.csv        → guardado")
print("  e_drugDB_smiles.csv → guardado")

# Resumen de calidad
print("\n  Resumen del modelo canónico (con SMILES):")
print(f"    Fármacos únicos           : {con_smiles['drug_id'].nunique():,}")
print(f"    Reacciones adversas únicas: {con_smiles['adverse_reaction'].nunique():,}")
print(f"    Pares fármaco-reacción    : {len(con_smiles):,}")


---
## 3. Perfilado de Datos — Antes de Limpieza (`perfilado.py`)

**Fase TDQM: MEDIR**

Se calcula la línea base de calidad sobre el modelo canónico `e_drugDB_smiles.csv` en 5 dimensiones:

| Dimensión | Descripción | Métrica |
|-----------|-------------|---------|
| **Completitud** | % campos obligatorios no nulos | SMILES, nombre, adverse_reaction, severity |
| **Unicidad** | Ausencia de duplicados | % pares (drug_id, adverse_reaction) únicos |
| **Consistencia** | Coherencia interna | source ∈ {SIDER+PubChem, SIDER_only} |
| **Exactitud** | Datos correctos | len(SMILES) > 2 |
| **Validez** | Conformidad con reglas dominio | SMILES parseable por RDKit |

> **ydata-profiling**: Se genera un reporte HTML interactivo completo. Usa una muestra de 20k filas para eficiencia de memoria.


In [ ]:
print("=" * 70)
print("PERFILADO — Fase MEDIR (antes de limpieza)")
print("=" * 70)

df = pd.read_csv(CLEAN / "e_drugDB_smiles.csv")
print(f"\n  {len(df):,} filas | {df['drug_id'].nunique():,} fármacos | "
      f"{df['adverse_reaction'].nunique():,} reacciones")

# ─── ydata-profiling (saltar si HTML ya existe) ───────────────────────────────
html_antes = PERFILES / "perfil_antes_limpieza.html"
if html_antes.exists():
    print(f"\n  ✓ perfil_antes_limpieza.html ya existe — omitiendo generación")
else:
    print("\n  Generando reporte HTML ydata-profiling (puede tardar ~1 min)...")
    from ydata_profiling import ProfileReport
    sample = df.sample(min(20_000, len(df)), random_state=42)
    profile = ProfileReport(
        sample,
        title="Farmacovigilancia — Modelo canónico e_drugDB (muestra 20k)",
        explorative=True, minimal=False
    )
    profile.to_file(html_antes)
    print(f"  Reporte guardado: {html_antes}")


In [ ]:
print("\n[Métricas de calidad — línea base]")

metrics = {}
metrics["Completitud — SMILES"]            = df["smiles"].notna().mean()
metrics["Completitud — nombre"]            = (df["name"].notna() &
                                               (df["name"].str.lower() != "desconocido")).mean()
metrics["Completitud — adverse_reaction"]  = df["adverse_reaction"].notna().mean()
metrics["Completitud — reaction_type"]     = df["reaction_type"].notna().mean()
metrics["Completitud — severity"]          = (df["severity"] != "unknown").mean()
metrics["Unicidad — pares únicos"]         = 1 - df.duplicated(["drug_id","adverse_reaction"]).mean()
metrics["Consistencia — source válido"]    = df["source"].isin(["SIDER+PubChem","SIDER_only"]).mean()
metrics["Consistencia — CID numérico"]     = pd.to_numeric(df["pubchem_cid"], errors="coerce").notna().mean()
metrics["Exactitud — SMILES longitud>2"]   = (df["smiles"].str.len() > 2).mean()

import numpy as _np
try:
    _smp = df["smiles"].dropna().sample(min(2000, len(df)), random_state=42)
    metrics["Validez — SMILES parseable"]  = _smp.apply(
        lambda s: Chem.MolFromSmiles(str(s)) is not None).mean()
except Exception:
    metrics["Validez — SMILES parseable"]  = float("nan")

print(f"\n  {'Dimensión':<45} {'Valor':>7}  {'Barra'}")
print("  " + "-"*70)
for k, v in metrics.items():
    bar = "█" * int(v * 20) if not _np.isnan(v) else "N/A"
    print(f"  {k:<45} {v*100:>6.1f}%  {bar}")

# Guardar CSV
pd.DataFrame({"metrica": list(metrics.keys()),
              "valor":   list(metrics.values())
}).to_csv(PERFILES / "metricas_calidad.csv", index=False)
print("\n  Métricas guardadas: metricas_calidad.csv")


In [ ]:
print("\n[Gráficas de perfilado]")
_fig_size = (10, 5)

# 01 — Top 20 reacciones adversas
top_reac = df["adverse_reaction"].value_counts().head(20).reset_index().rename(columns={"count":"n"})
fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=top_reac, x="n", y="adverse_reaction", ax=ax, color="#4C72B0")
ax.set_title("Top 20 reacciones adversas más frecuentes (# fármacos)")
ax.set_xlabel("Número de fármacos"); ax.set_ylabel("")
for p in ax.patches:
    ax.text(p.get_width()+50, p.get_y()+p.get_height()/2,
            f"{int(p.get_width()):,}", va="center", fontsize=8)
plt.tight_layout()
guardar(fig, PERFILES / "01_top20_reacciones.png")

# 02 — Distribución de reacciones por fármaco
reac_per_drug = df.groupby("drug_id")["adverse_reaction"].nunique()
fig, ax = plt.subplots(figsize=_fig_size)
ax.hist(reac_per_drug, bins=50, color="#DD8452", edgecolor="white")
ax.axvline(reac_per_drug.median(), color="red", linestyle="--",
           label=f"Mediana={reac_per_drug.median():.0f}")
ax.set_title("Distribución de reacciones adversas por fármaco")
ax.set_xlabel("Número de reacciones distintas"); ax.set_ylabel("Número de fármacos")
ax.legend(); plt.tight_layout()
guardar(fig, PERFILES / "02_dist_reacciones_por_farmaco.png")

# 03 — Completitud por columna
completitud = df.isnull().mean().sort_values()
fig, ax = plt.subplots(figsize=_fig_size)
colors_c = ["#2ca02c" if v == 0 else "#d62728" for v in completitud]
bars = ax.barh(completitud.index, (1-completitud)*100, color=colors_c)
ax.set_xlim(0, 110); ax.set_xlabel("% de valores no nulos")
ax.set_title("Completitud por columna")
for bar, val in zip(bars, (1-completitud)*100):
    ax.text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2,
            f"{val:.1f}%", va="center", fontsize=9)
plt.tight_layout()
guardar(fig, PERFILES / "03_completitud_columnas.png")

# 04 — Longitud de SMILES boxplot
df["smiles_len"] = df["smiles"].str.len()
top10_reac = df["adverse_reaction"].value_counts().head(10).index
sub = df[df["adverse_reaction"].isin(top10_reac)]
fig, ax = plt.subplots(figsize=(12, 5))
sns.boxplot(data=sub, x="adverse_reaction", y="smiles_len", order=top10_reac, color="#55A868")
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right", fontsize=8)
ax.set_title("Longitud de SMILES para los 10 efectos adversos más frecuentes")
ax.set_xlabel(""); ax.set_ylabel("Longitud del SMILES")
plt.tight_layout()
guardar(fig, PERFILES / "04_smiles_len_boxplot.png")

# 05 — Métricas de calidad radar/barras
met_names  = list(metrics.keys())
met_values = [v*100 if not _np.isnan(v) else 0 for v in metrics.values()]
fig, ax = plt.subplots(figsize=(11, 5))
colors_met = ["#4C72B0" if v >= 95 else "#DD8452" if v >= 80 else "#C44E52"
              for v in met_values]
bars = ax.barh(met_names, met_values, color=colors_met)
ax.set_xlim(0, 110); ax.axvline(100, color="gray", linestyle="--", lw=0.8)
ax.set_xlabel("Porcentaje (%)"); ax.set_title("Métricas de calidad — TDQM Fase: MEDIR")
for bar, val in zip(bars, met_values):
    ax.text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2,
            f"{val:.1f}%", va="center", fontsize=8)
plt.tight_layout()
guardar(fig, PERFILES / "05_metricas_calidad.png")

# 06 — Top 20 fármacos con más reacciones
top_drugs = (df.groupby("name")["adverse_reaction"].nunique()
             .sort_values(ascending=False).head(20)
             .reset_index().rename(columns={"adverse_reaction": "n_reacciones"}))
fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(data=top_drugs, x="n_reacciones", y="name", ax=ax, color="#8172B2")
ax.set_title("Top 20 fármacos con más reacciones adversas reportadas")
ax.set_xlabel("# reacciones adversas únicas"); ax.set_ylabel("")
plt.tight_layout()
guardar(fig, PERFILES / "06_top20_farmacos.png")

# 07 — Distribución por fuente (pie)
src_counts = df["source"].value_counts()
fig, ax = plt.subplots(figsize=(6, 5))
ax.pie(src_counts, labels=src_counts.index, autopct="%1.1f%%",
       colors=["#4C72B0","#DD8452"], startangle=90)
ax.set_title("Distribución por fuente de datos")
plt.tight_layout()
guardar(fig, PERFILES / "07_fuente_pie.png")

print("\n  Gráficas generadas: 01–07")


---
## 4. Limpieza de Datos (`limpieza.py`)

**Fase TDQM: MEJORAR**

Se aplican 5 operaciones de mejora sobre `e_drugDB_smiles.csv`:

1. **Imputación de severity** — desde `meddra_freq.tsv` (291,632 filas de frecuencias). Mapeo: very common→very_common, common→common, uncommon→uncommon, rare→rare, postmarketing→postmarketing. Se toma la severidad más alta por par (stitch_flat, reaction_type).
2. **Validación de SMILES con RDKit** — `MolFromSmiles()` + `SanitizeMol()` (valencias y aromaticidad). Se canonicalizan con `MolToSmiles()`.
3. **Normalización de nombres** — Strip, colapso de espacios, Title Case, corrección de acrónimos médicos (ASA, PGE2, LH, FSH, ACTH).
4. **Normalización de adverse_reaction** — Strip + Title Case sobre todos los términos MedDRA.
5. **Detección de outliers IQR** — En `n_reac` por fármaco. Outliers conservados (son señales clínicas reales: Pregabalin=742, Aripiprazole=741).


In [ ]:
print("=" * 70)
print("LIMPIEZA — Fase MEJORAR")
print("=" * 70)

df = pd.read_csv(CLEAN / "e_drugDB_smiles.csv")
filas_iniciales = len(df)
print(f"\n  {filas_iniciales:,} filas cargadas")

antes = {
    "filas"              : filas_iniciales,
    "severity_known_pct" : (df["severity"] != "unknown").mean() * 100,
    "smiles_validos_pct" : 0.0,
    "nombres_limpios_pct": 0.0,
    "duplicados"         : df.duplicated(["drug_id", "adverse_reaction"]).sum(),
}


In [ ]:
print("\n[1] Imputando severity desde meddra_freq.tsv...")

cols_freq = ["stitch_flat","stitch_stereo","umls_found","placebo",
             "freq_str","freq_low","freq_hi","meddra_type","umls_meddra","side_effect_name"]
freq = pd.read_csv(RAW / "meddra_freq.tsv", sep="\t", header=None, names=cols_freq)
freq_pt = freq[freq["meddra_type"] == "PT"].copy()

SEVERITY_MAP = {
    "very common": "very_common", "common": "common",   "frequent": "common",
    "uncommon":    "uncommon",    "infrequent": "uncommon",
    "rare":        "rare",        "very rare": "very_rare",
    "postmarketing": "postmarketing",
}
ORDER_SEV = {"very_common":5, "common":4, "uncommon":3, "rare":2,
             "very_rare":1, "postmarketing":0}

def clasificar_frecuencia(s):
    if pd.isna(s):
        return None
    s = str(s).strip().lower()
    if s in SEVERITY_MAP:
        return SEVERITY_MAP[s]
    m = re.match(r"^(\d+(?:\.\d+)?)\s*%?$", s)
    if m:
        pct = float(m.group(1))
        if pct >= 10:  return "very_common"
        if pct >= 1:   return "common"
        if pct >= 0.1: return "uncommon"
        if pct > 0:    return "rare"
    return None

freq_pt["severity_calc"] = freq_pt["freq_str"].apply(clasificar_frecuencia)
freq_pt["sev_rank"] = freq_pt["severity_calc"].map(ORDER_SEV).fillna(-1)
best_sev = (
    freq_pt.dropna(subset=["severity_calc"])
    .sort_values("sev_rank", ascending=False)
    .drop_duplicates(subset=["stitch_flat", "umls_meddra"])
    [["stitch_flat", "umls_meddra", "severity_calc"]]
    .rename(columns={"umls_meddra": "reaction_type"})
)

before_sev = (df["severity"] == "unknown").sum()
df = df.merge(best_sev, on=["stitch_flat", "reaction_type"], how="left")
df["severity"] = df["severity_calc"].fillna(df["severity"])
df.drop(columns=["severity_calc"], inplace=True)
after_sev = (df["severity"] == "unknown").sum()

print(f"  Sin dato antes    : {before_sev:,}")
print(f"  Imputados         : {before_sev - after_sev:,}")
print(f"  Aún desconocidos  : {after_sev:,}  ({100*after_sev/len(df):.1f}%)")
print("  Distribución:")
for cat, n in df["severity"].value_counts().items():
    print(f"    {cat:<15}: {n:>6,}  ({100*n/len(df):.1f}%)")


In [ ]:
print("\n[2] Validando SMILES con RDKit + canonicalizando...")

def smiles_ok(s):
    try:
        mol = Chem.MolFromSmiles(str(s))
        if mol is None or mol.GetNumAtoms() == 0:
            return False
        Chem.SanitizeMol(mol)
        return True
    except Exception:
        return False

def canonicalizar_smiles(s):
    try:
        mol = Chem.MolFromSmiles(str(s))
        if mol is not None:
            return Chem.MolToSmiles(mol)
    except Exception:
        pass
    return s

df["smiles_valido"] = df["smiles"].apply(smiles_ok)
n_invalidos = (~df["smiles_valido"]).sum()
antes["smiles_validos_pct"] = df["smiles_valido"].mean() * 100

print(f"  SMILES válidos   : {df['smiles_valido'].sum():,}  ({df['smiles_valido'].mean()*100:.2f}%)")
print(f"  SMILES inválidos : {n_invalidos:,}")

if n_invalidos > 0:
    print("  Ejemplos de SMILES inválidos:")
    for _, row in df[~df["smiles_valido"]].head(5).iterrows():
        print(f"    CID {row['pubchem_cid']} | {str(row['smiles'])[:60]}")
    df = df[df["smiles_valido"]].copy()

df.drop(columns=["smiles_valido"], inplace=True)

print("  Canonicalizando SMILES...")
df["smiles"] = df["smiles"].apply(canonicalizar_smiles)
print(f"  Canonicalización completada: {df['smiles'].notna().sum():,} registros")


In [ ]:
# ─── Normalización de nombres ─────────────────────────────────────────────────
print("\n[3] Normalizando nombres de medicamentos...")

def normalizar_nombre(s):
    if pd.isna(s) or str(s).strip().lower() in ("nan","none","desconocido",""):
        return None
    s = str(s).strip()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^\w\s\-\(\)\[\]\,\.\']", "", s)
    s = s.title()
    for pat, rep in [("\bAsa\b","ASA"),("\bPge2\b","PGE2"),
                     ("\bLh\b","LH"),("\bFsh\b","FSH"),("\bActh\b","ACTH")]:
        s = re.sub(pat, rep, s)
    return s

nombres_antes = df["name"].copy()
df["name"]    = df["name"].apply(normalizar_nombre)
cambiados     = (nombres_antes != df["name"]).sum()
antes["nombres_limpios_pct"] = (nombres_antes.apply(normalizar_nombre) == nombres_antes).mean() * 100
df["name"] = df["name"].fillna("Desconocido")
print(f"  Nombres modificados: {cambiados:,}")

# ─── Normalización de adverse_reaction ────────────────────────────────────────
print("\n[4] Normalizando términos MedDRA...")
def normalizar_reaccion(s):
    if pd.isna(s): return None
    return re.sub(r"\s+", " ", str(s).strip()).title()

df["adverse_reaction"] = df["adverse_reaction"].apply(normalizar_reaccion)
print(f"  Únicos tras normalización: {df['adverse_reaction'].nunique():,}")

# ─── Outliers por IQR ─────────────────────────────────────────────────────────
print("\n[5] Detección de outliers por IQR...")
reac_por_drug = df.groupby("drug_id")["adverse_reaction"].nunique().rename("n_reac")
Q1, Q3 = reac_por_drug.quantile(0.25), reac_por_drug.quantile(0.75)
IQR    = Q3 - Q1
high   = Q3 + 1.5 * IQR
outliers_high = reac_por_drug[reac_por_drug > high]
print(f"  Q1={Q1:.0f}  Q3={Q3:.0f}  IQR={IQR:.0f}  Límite sup={high:.0f}")
print(f"  Fármacos sobre límite: {len(outliers_high):,}")
if len(outliers_high) > 0:
    print("  Top 5 outliers:")
    for cid, n in outliers_high.sort_values(ascending=False).head(5).items():
        name = df[df["drug_id"]==cid]["name"].iloc[0]
        print(f"    {name:<30} {n:>5,} reacciones")
print("  Decisión: outliers CONSERVADOS — señales clínicas reales")

df = df.join(reac_por_drug, on="drug_id")

# ─── Guardar dataset limpio ────────────────────────────────────────────────────
print("\n[6] Guardando e_drugDB_clean.csv...")
cols_fin = ["drug_id","name","smiles","adverse_reaction","reaction_type",
            "severity","source","pubchem_cid","stitch_flat","n_reac"]
df_clean = df[cols_fin].copy()
df_clean.to_csv(CLEAN / "e_drugDB_clean.csv", index=False)
size_kb = (CLEAN/"e_drugDB_clean.csv").stat().st_size / 1024
print(f"  {len(df_clean):,} filas  |  {size_kb:.0f} KB")


In [ ]:
print("\n[7] Generando gráficas comparativas antes/después de limpieza...")

despues = {
    "severity_known_pct" : (df_clean["severity"] != "unknown").mean() * 100,
    "smiles_validos_pct" : 100.0,
    "nombres_limpios_pct": (df_clean["name"] != "Desconocido").mean() * 100,
}

# 08 — Completitud antes vs después
dims_comp  = ["severity\nconocido", "SMILES\nválidos", "nombres\nlimpios"]
vals_antes = [antes["severity_known_pct"], antes["smiles_validos_pct"], antes["nombres_limpios_pct"]]
vals_desp  = [despues["severity_known_pct"], despues["smiles_validos_pct"], despues["nombres_limpios_pct"]]
x = _np.arange(len(dims_comp)); w = 0.35
fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x-w/2, vals_antes, w, label="Antes limpieza",  color="#DD8452")
b2 = ax.bar(x+w/2, vals_desp,  w, label="Después limpieza", color="#4C72B0")
ax.set_ylim(0, 115); ax.set_xticks(x); ax.set_xticklabels(dims_comp)
ax.set_ylabel("Porcentaje (%)"); ax.set_title("Completitud antes vs después de limpieza")
ax.legend()
for bar in list(b1)+list(b2):
    h = bar.get_height()
    ax.text(bar.get_x()+bar.get_width()/2, h+1,
            f"{h:.1f}%", ha="center", va="bottom", fontsize=8)
plt.tight_layout()
guardar(fig, PERFILES / "08_antes_despues_completitud.png")

# 09 — Distribución de severity
ORDER_SEV_PLOT = ["very_common","common","uncommon","rare","very_rare","postmarketing","unknown"]
sev_counts = df_clean["severity"].value_counts().reindex(ORDER_SEV_PLOT, fill_value=0)
fig, ax = plt.subplots(figsize=(10, 5))
colors_s = ["#d62728","#ff7f0e","#2ca02c","#1f77b4","#9467bd","#8c564b","#c7c7c7"]
bars = ax.bar(sev_counts.index, sev_counts.values, color=colors_s)
ax.set_title("Distribución de severidad/frecuencia de reacciones adversas")
ax.set_xlabel("Categoría de frecuencia"); ax.set_ylabel("# pares fármaco-reacción")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{int(v):,}"))
for bar, val in zip(bars, sev_counts.values):
    if val > 0:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+200,
                f"{val:,}", ha="center", va="bottom", fontsize=8)
plt.tight_layout()
guardar(fig, PERFILES / "09_severidad_distribucion.png")

# 10 — Outliers boxplot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].boxplot(reac_por_drug.values, patch_artist=True,
                boxprops=dict(facecolor="#4C72B0", alpha=0.7))
axes[0].axhline(high, color="red", linestyle="--",
                label=f"Límite IQR ({high:.0f})")
axes[0].set_title("Reacciones por fármaco — boxplot")
axes[0].set_ylabel("# reacciones adversas"); axes[0].legend(fontsize=8)
axes[1].hist(reac_por_drug.values, bins=60, color="#55A868", edgecolor="white")
axes[1].axvline(high, color="red", linestyle="--")
axes[1].set_title("Distribución reacciones por fármaco")
axes[1].set_xlabel("# reacciones"); axes[1].set_ylabel("# fármacos")
plt.tight_layout()
guardar(fig, PERFILES / "10_outliers_iqr.png")

# 11 — Heatmap top-15 fármacos × top-15 reacciones
top_drugs_names = (df_clean.groupby("name")["adverse_reaction"].nunique()
                   .sort_values(ascending=False).head(15).index)
top_reac_names  = df_clean["adverse_reaction"].value_counts().head(15).index
pivot = (df_clean[df_clean["name"].isin(top_drugs_names) &
                  df_clean["adverse_reaction"].isin(top_reac_names)]
         .assign(present=1)
         .pivot_table(index="name", columns="adverse_reaction",
                      values="present", aggfunc="max", fill_value=0))
fig, ax = plt.subplots(figsize=(14, 7))
sns.heatmap(pivot, ax=ax, cmap="Blues", linewidths=0.4,
            cbar_kws={"label":"Reacción reportada"})
ax.set_title("Presencia de reacciones: top 15 fármacos × top 15 reacciones")
plt.xticks(rotation=35, ha="right", fontsize=7); plt.yticks(fontsize=7)
plt.tight_layout()
guardar(fig, PERFILES / "11_heatmap_farmaco_reaccion.png")

print("  Gráficas 08–11 generadas")


---
## 5. Perfilado Después de Limpieza — Comparación Antes/Después

**Fase TDQM: MEDIR (post-mejora)**

Se re-calculan las mismas métricas sobre `e_drugDB_clean.csv` y se comparan con la línea base.
El cambio más significativo es la imputación de **severity**: de 0% a 41.4% de cobertura.


In [ ]:
print("=" * 70)
print("PERFILADO DESPUÉS — Fase MEDIR (post-limpieza)")
print("=" * 70)

df_clean = pd.read_csv(CLEAN / "e_drugDB_clean.csv")
print(f"\n  {len(df_clean):,} filas | {df_clean['drug_id'].nunique():,} fármacos")

# ─── ydata-profiling post-limpieza ────────────────────────────────────────────
html_desp = PERFILES / "perfil_despues_limpieza.html"
if html_desp.exists():
    print(f"\n  ✓ perfil_despues_limpieza.html ya existe — omitiendo")
else:
    from ydata_profiling import ProfileReport
    print("\n  Generando reporte HTML post-limpieza...")
    sample_clean = df_clean.sample(min(20_000, len(df_clean)), random_state=42)
    profile_clean = ProfileReport(
        sample_clean,
        title="Farmacovigilancia — e_drugDB DESPUÉS de limpieza (muestra 20k)",
        explorative=True, minimal=False
    )
    profile_clean.to_file(html_desp)
    print(f"  Guardado: {html_desp}")

# ─── Métricas post-limpieza ───────────────────────────────────────────────────
metrics_clean = {}
metrics_clean["Completitud — SMILES"]           = df_clean["smiles"].notna().mean()
metrics_clean["Completitud — nombre"]           = (
    df_clean["name"].notna() & (df_clean["name"].str.lower() != "desconocido")).mean()
metrics_clean["Completitud — adverse_reaction"] = df_clean["adverse_reaction"].notna().mean()
metrics_clean["Completitud — reaction_type"]    = df_clean["reaction_type"].notna().mean()
metrics_clean["Completitud — severity"]         = (df_clean["severity"] != "unknown").mean()
metrics_clean["Unicidad — pares únicos"]        = 1 - df_clean.duplicated(["drug_id","adverse_reaction"]).mean()
metrics_clean["Consistencia — source válido"]   = df_clean["source"].isin(["SIDER+PubChem","SIDER_only"]).mean()
metrics_clean["Consistencia — CID numérico"]    = pd.to_numeric(df_clean["pubchem_cid"],errors="coerce").notna().mean()
metrics_clean["Exactitud — SMILES longitud>2"]  = (df_clean["smiles"].str.len() > 2).mean()
try:
    _smp2 = df_clean["smiles"].dropna().sample(min(2000,len(df_clean)),random_state=42)
    metrics_clean["Validez — SMILES parseable"] = _smp2.apply(
        lambda s: Chem.MolFromSmiles(str(s)) is not None).mean()
except Exception:
    metrics_clean["Validez — SMILES parseable"] = float("nan")

# ─── Tabla comparativa ────────────────────────────────────────────────────────
print(f"\n  {'Métrica':<45} {'ANTES':>8}  {'DESPUÉS':>8}  {'Δ':>8}")
print("  " + "-"*75)
comp_rows = []
for k in metrics:
    v_a   = metrics[k]
    v_d   = metrics_clean.get(k, float("nan"))
    delta = v_d - v_a if not (_np.isnan(v_a) or _np.isnan(v_d)) else float("nan")
    signo = "▲" if delta > 0.001 else ("▼" if delta < -0.001 else "=")
    a_s   = f"{v_a*100:>6.1f}%" if not _np.isnan(v_a) else "   N/A"
    d_s   = f"{v_d*100:>6.1f}%" if not _np.isnan(v_d) else "   N/A"
    dt_s  = f"{signo}{abs(delta)*100:>4.1f}%" if not _np.isnan(delta) else "    N/A"
    print(f"  {k:<45} {a_s}   {d_s}   {dt_s}")
    comp_rows.append({"metrica":k,"antes":v_a,"despues":v_d,"delta":delta})

pd.DataFrame(comp_rows).to_csv(PERFILES/"comparacion_antes_despues.csv", index=False)

# ─── Resumen CSV antes/después ────────────────────────────────────────────────
resumen = [
    ("Filas totales",          f"{antes['filas']:,}", f"{len(df_clean):,}"),
    ("Severity conocida (%)",  f"{antes['severity_known_pct']:.1f}%",
                                f"{despues['severity_known_pct']:.1f}%"),
    ("SMILES válidos (%)",     f"{antes['smiles_validos_pct']:.1f}%",
                                f"{despues['smiles_validos_pct']:.1f}%"),
    ("Nombres limpios (%)",    f"{antes['nombres_limpios_pct']:.1f}%",
                                f"{despues['nombres_limpios_pct']:.1f}%"),
    ("Duplicados drug+reac",   f"{antes['duplicados']:,}", "0"),
]
pd.DataFrame(resumen, columns=["metrica","antes","despues"]).to_csv(
    PERFILES/"resumen_antes_despues.csv", index=False)
print("\n  CSVs guardados: comparacion_antes_despues.csv, resumen_antes_despues.csv")


In [ ]:
# ─── Gráficas comparativas ────────────────────────────────────────────────────
labels   = [k.split(" — ")[1] if " — " in k else k for k in metrics]
v_a_list = [metrics[k]*100 if not _np.isnan(metrics[k]) else 0 for k in metrics]
v_d_list = [metrics_clean.get(k,0)*100 if not _np.isnan(metrics_clean.get(k,0)) else 0
            for k in metrics]

x_pos = _np.arange(len(labels)); w = 0.38
fig, ax = plt.subplots(figsize=(13, 6))
b1 = ax.barh(x_pos+w/2, v_a_list, w, label="Antes limpieza",  color="#DD8452", alpha=0.85)
b2 = ax.barh(x_pos-w/2, v_d_list, w, label="Después limpieza", color="#4C72B0", alpha=0.85)
ax.set_yticks(x_pos); ax.set_yticklabels(labels, fontsize=9)
ax.set_xlim(0, 115); ax.axvline(100, color="gray", linestyle="--", lw=0.8)
ax.set_xlabel("Porcentaje (%)"); ax.legend(loc="lower right")
ax.set_title("Perfilado ANTES vs DESPUÉS de limpieza — dimensiones de calidad TDQM")
for bar in list(b1)+list(b2):
    h = bar.get_width()
    if h > 1:
        ax.text(h+0.3, bar.get_y()+bar.get_height()/2,
                f"{h:.1f}%", va="center", fontsize=7.5)
plt.tight_layout()
guardar(fig, PERFILES / "12_comparacion_antes_despues.png")

# Mejora neta por dimensión
deltas   = [v_d_list[i]-v_a_list[i] for i in range(len(labels))]
colors_d = ["#2ca02c" if d>0 else "#d62728" if d<0 else "#aec7e8" for d in deltas]
fig, ax = plt.subplots(figsize=(12, 5))
bars_d = ax.barh(labels, deltas, color=colors_d, edgecolor="white")
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("Mejora en puntos porcentuales (Δ%)")
ax.set_title("Mejora neta por dimensión de calidad tras limpieza (TDQM: MEJORAR)")
for bar, val in zip(bars_d, deltas):
    xpos = bar.get_width()+(0.3 if val>=0 else -0.3)
    ha   = "left" if val>=0 else "right"
    ax.text(xpos, bar.get_y()+bar.get_height()/2,
            f"{val:+.1f}%", va="center", ha=ha, fontsize=8)
plt.tight_layout()
guardar(fig, PERFILES / "13_mejora_neta_dimensiones.png")

print("Gráficas 12–13 generadas")


---
## 6. Análisis — Carga y Propiedades Moleculares (`análisis.py`)

**Fase TDQM: ANALIZAR**

Se carga el dataset limpio y se calculan propiedades moleculares con RDKit:
- **MW** — Peso molecular (Da)
- **logP** — Lipofilicidad (Crippen)
- **HBD/HBA** — Donadores/Aceptores de enlace de hidrógeno
- **n_rings** — Número de anillos
- **n_atoms** — Número de átomos pesados
- **TPSA** — Topological Polar Surface Area (Å²)

Estas propiedades permiten evaluar las **Reglas de Lipinski** (Rule of 5) y analizar el espacio farmacológico de los 1,556 fármacos.


In [ ]:
print("=" * 70)
print("ANÁLISIS — Fase ANALIZAR")
print("=" * 70)

# ─── Cargar dataset limpio ────────────────────────────────────────────────────
df = pd.read_csv(CLEAN / "e_drugDB_clean.csv")
print(f"\n  {len(df):,} filas | {df['drug_id'].nunique():,} fármacos | "
      f"{df['adverse_reaction'].nunique():,} reacciones")

# Fuentes raw para consultas individuales
sider_raw  = pd.read_csv(RAW / "sider_se_pt.csv")
drug_names = pd.read_csv(RAW / "sider_drug_names.csv")
smiles_raw = pd.read_csv(RAW / "pubchem_smiles.csv")

# ─── Propiedades moleculares (RDKit) ─────────────────────────────────────────
print("\n[Mol] Calculando propiedades moleculares con RDKit...")
drug_smiles = df[["drug_id","name","smiles"]].drop_duplicates("drug_id").copy()
rows = []
for _, r in drug_smiles.iterrows():
    try:
        mol = Chem.MolFromSmiles(r["smiles"])
        if mol is None: continue
        rows.append({
            "drug_id"  : r["drug_id"],
            "name"     : r["name"],
            "smiles"   : r["smiles"],
            "MW"       : Descriptors.MolWt(mol),
            "logP"     : Descriptors.MolLogP(mol),
            "HBD"      : rdMolDescriptors.CalcNumHBD(mol),
            "HBA"      : rdMolDescriptors.CalcNumHBA(mol),
            "n_rings"  : rdMolDescriptors.CalcNumRings(mol),
            "n_atoms"  : mol.GetNumAtoms(),
            "TPSA"     : Descriptors.TPSA(mol),
        })
    except Exception:
        pass

mol_df = pd.DataFrame(rows)
print(f"  Propiedades calculadas: {len(mol_df):,} fármacos")

# Unir al df principal
df = df.merge(mol_df[["drug_id","MW","logP","HBD","HBA","n_rings","n_atoms","TPSA"]],
              on="drug_id", how="left")

# drug_stats — una fila por fármaco con propiedades moleculares + n_reac
drug_stats = (
    df.groupby("drug_id")
    .agg(n_reac=("adverse_reaction","nunique"),
         MW=("MW","first"), logP=("logP","first"), name=("name","first"))
    .dropna(subset=["MW"])
    .reset_index()
)
print(f"  drug_stats: {len(drug_stats):,} fármacos con propiedades moleculares")


---
## 7. Sección A — Consultas sobre SIDER (5 consultas)

Análisis sobre la fuente SIDER de forma individual, antes de la integración:
- **A1** Top 15 reacciones más reportadas (Nausea, Headache, Dizziness lideran)
- **A2** Distribución de reacciones por fármaco (sesgo a la derecha, media≈98, mediana≈75)
- **A3** Top 15 fármacos con más reacciones (Pregabalin=742, Aripiprazole=741)
- **A4** Scatter de reacciones por CID (no hay patrón por orden de CID)
- **A5** Heatmap co-ocurrencia top-10 reacciones (Nausea y Vomiting altamente co-ocurrentes)


In [ ]:
print("─" * 70)
print("SECCIÓN A — Consultas sobre SIDER")
print("─" * 70)

# A1 — Top 15 reacciones adversas (SIDER)
print("\n  A1: Top 15 reacciones adversas en SIDER")
top_reac_a = (sider_raw["side_effect_name"].value_counts().head(15)
              .reset_index().rename(columns={"count":"n"}))
fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(data=top_reac_a, x="n", y="side_effect_name", ax=ax, palette="Blues_r")
ax.set_title("A1 — Top 15 reacciones adversas más reportadas en SIDER 4.1")
ax.set_xlabel("Número de fármacos asociados"); ax.set_ylabel("")
for p in ax.patches:
    ax.text(p.get_width()+3, p.get_y()+p.get_height()/2,
            f"{int(p.get_width()):,}", va="center", fontsize=8)
plt.tight_layout(); guardar(fig, GRAFICAS/"A1_top15_reacciones_sider.png")

# A2 — Distribución de reacciones por fármaco
print("  A2: Distribución de reacciones por fármaco")
reac_x_drug = sider_raw.groupby("pubchem_cid")["side_effect_name"].nunique()
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(reac_x_drug, bins=60, color="#4C72B0", edgecolor="white", alpha=0.85)
ax.axvline(reac_x_drug.median(), color="red",    linestyle="--",
           label=f"Mediana={reac_x_drug.median():.0f}")
ax.axvline(reac_x_drug.mean(),   color="orange", linestyle="--",
           label=f"Media={reac_x_drug.mean():.0f}")
ax.set_title("A2 — Distribución de reacciones adversas por fármaco (SIDER)")
ax.set_xlabel("# reacciones distintas"); ax.set_ylabel("# fármacos"); ax.legend()
plt.tight_layout(); guardar(fig, GRAFICAS/"A2_dist_reacciones_farmaco_sider.png")

# A3 — Top 15 fármacos por # reacciones
print("  A3: Top 15 fármacos con más reacciones (SIDER)")
top_drugs_sider = (sider_raw.merge(drug_names, on="stitch_flat", how="left")
                   .groupby("drug_name")["side_effect_name"].nunique()
                   .sort_values(ascending=False).head(15)
                   .reset_index().rename(columns={"side_effect_name":"n"}))
fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(data=top_drugs_sider, x="n", y="drug_name", ax=ax, palette="Oranges_r")
ax.set_title("A3 — Top 15 fármacos con más reacciones adversas (SIDER 4.1)")
ax.set_xlabel("# reacciones adversas únicas"); ax.set_ylabel("")
plt.tight_layout(); guardar(fig, GRAFICAS/"A3_top15_farmacos_sider.png")

# A4 — Scatter CID vs #reacciones
print("  A4: Scatter reacciones por CID")
vals_a4 = reac_x_drug.sort_values(ascending=False).values
fig, ax = plt.subplots(figsize=(11, 5))
ax.scatter(range(len(vals_a4)), vals_a4, s=4, alpha=0.5, color="#55A868")
ax.fill_between(range(len(vals_a4)), vals_a4, alpha=0.1, color="#55A868")
ax.set_title("A4 — Reacciones adversas únicas por fármaco (SIDER, ordenado desc)")
ax.set_xlabel("Fármaco (ordenado por # reacciones)"); ax.set_ylabel("# reacciones únicas")
plt.tight_layout(); guardar(fig, GRAFICAS/"A4_scatter_reacciones_cid.png")

# A5 — Heatmap co-ocurrencia top-10 reacciones
print("  A5: Co-ocurrencia top-10 reacciones")
top10_se = sider_raw["side_effect_name"].value_counts().head(10).index
sub_se   = sider_raw[sider_raw["side_effect_name"].isin(top10_se)]
pivot_se = sub_se.assign(v=1).pivot_table(
    index="pubchem_cid", columns="side_effect_name", values="v",
    aggfunc="max", fill_value=0)
co_se = pivot_se.T.dot(pivot_se)
_np.fill_diagonal(co_se.values, 0)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(co_se, ax=ax, cmap="YlOrRd", annot=True, fmt="d",
            linewidths=0.5, cbar_kws={"label":"# fármacos en común"})
ax.set_title("A5 — Co-ocurrencia de las 10 reacciones más frecuentes (SIDER)")
plt.xticks(rotation=35, ha="right", fontsize=8); plt.yticks(fontsize=8)
plt.tight_layout(); guardar(fig, GRAFICAS/"A5_coocurrencia_reacciones.png")

print("\n  Sección A completa — 5 gráficas generadas")


---
## 8. Sección B — Consultas sobre PubChem (5 consultas)

Análisis del espacio farmacológico de los 1,556 fármacos con estructuras moleculares:
- **B1** Distribución de peso molecular (MW concentrado 200–600 Da)
- **B2** Espacio de Lipinski (MW vs logP) — mayoría cumple Rule of 5
- **B3** Distribución de anillos aromáticos/alicíclicos
- **B4** Violin HBD + HBA (donadores/aceptores de enlace H)
- **B5** TPSA vs número de átomos (correlación positiva)

### Regla de Lipinski (Rule of 5)
Un fármaco oral tiene buena absorción si cumple: MW ≤ 500 Da, logP ≤ 5, HBD ≤ 5, HBA ≤ 10. El cumplimiento de estas reglas es un proxy de la "drug-likeness" de la molécula.


In [ ]:
print("─" * 70)
print("SECCIÓN B — Consultas sobre PubChem")
print("─" * 70)

# B1 — Distribución de peso molecular
print("\n  B1: Distribución de peso molecular")
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(mol_df["MW"], bins=60, kde=True, ax=ax, color="#4C72B0")
ax.axvline(500, color="red", linestyle="--", label="Lipinski MW=500")
ax.set_title("B1 — Distribución de peso molecular de fármacos (PubChem/RDKit)")
ax.set_xlabel("Peso molecular (Da)"); ax.set_ylabel("# fármacos"); ax.legend()
plt.tight_layout(); guardar(fig, GRAFICAS/"B1_dist_peso_molecular.png")

# B2 — logP vs MW (Lipinski)
print("  B2: logP vs MW — Reglas de Lipinski")
fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(mol_df["MW"], mol_df["logP"], s=15, alpha=0.5,
                c=mol_df["n_rings"], cmap="viridis")
plt.colorbar(sc, ax=ax, label="# anillos")
ax.axvline(500, color="red",    linestyle="--", alpha=0.6, label="MW≤500")
ax.axhline(5,   color="orange", linestyle="--", alpha=0.6, label="logP≤5")
lipinski_ok = ((mol_df["MW"]<=500) & (mol_df["logP"]<=5)).mean()
ax.text(0.02, 0.96, f"Cumplen Lipinski: {lipinski_ok*100:.0f}%",
        transform=ax.transAxes, fontsize=9, va="top",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))
ax.set_title("B2 — Espacio farmacológico: MW vs logP (Reglas de Lipinski)")
ax.set_xlabel("Peso molecular (Da)"); ax.set_ylabel("logP"); ax.legend(fontsize=8)
plt.tight_layout(); guardar(fig, GRAFICAS/"B2_lipinski_scatter.png")

# B3 — Distribución de anillos
print("  B3: Distribución de número de anillos")
ring_counts = mol_df["n_rings"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(ring_counts.index, ring_counts.values, color="#DD8452", edgecolor="white")
ax.set_title("B3 — Distribución de número de anillos (PubChem/RDKit)")
ax.set_xlabel("# anillos"); ax.set_ylabel("# fármacos")
for x_b, y_b in zip(ring_counts.index, ring_counts.values):
    ax.text(x_b, y_b+1, str(y_b), ha="center", fontsize=8)
plt.tight_layout(); guardar(fig, GRAFICAS/"B3_dist_anillos.png")

# B4 — Violin HBD + HBA
print("  B4: Distribución HBD y HBA")
hb_data = pd.DataFrame({
    "Tipo": ["Donantes (HBD)"]*len(mol_df) + ["Aceptores (HBA)"]*len(mol_df),
    "Valor": list(mol_df["HBD"]) + list(mol_df["HBA"])
})
fig, ax = plt.subplots(figsize=(9, 5))
sns.violinplot(data=hb_data, x="Tipo", y="Valor", ax=ax,
               palette=["#4C72B0","#DD8452"], inner="box")
ax.set_title("B4 — Distribución de HBD y HBA (PubChem/RDKit)")
ax.set_ylabel("# grupos"); ax.set_xlabel("")
plt.tight_layout(); guardar(fig, GRAFICAS/"B4_violin_hbd_hba.png")

# B5 — TPSA vs # átomos
print("  B5: TPSA vs número de átomos")
fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(mol_df["n_atoms"], mol_df["TPSA"], s=15, alpha=0.5,
                c=mol_df["logP"], cmap="RdYlGn")
plt.colorbar(sc, ax=ax, label="logP")
ax.axhline(140, color="red", linestyle="--", alpha=0.7, label="TPSA=140 (permeabilidad)")
ax.set_title("B5 — TPSA vs número de átomos pesados (PubChem/RDKit)")
ax.set_xlabel("# átomos pesados"); ax.set_ylabel("TPSA (Å²)"); ax.legend(fontsize=8)
plt.tight_layout(); guardar(fig, GRAFICAS/"B5_tpsa_natoms.png")

print("\n  Sección B completa — 5 gráficas generadas")


---
## 9. Sección C — Consultas Cross-Source: SIDER + PubChem (10 consultas)

Cruce de información molecular (PubChem) con perfiles de reacciones adversas (SIDER).
Cada consulta aborda una dimensión de calidad o una pregunta científica:

| # | Consulta | Dimensión TDQM |
|---|----------|----------------|
| C1 | Completitud SMILES vs # reacciones | P1 — Completitud |
| C2 | Peso molecular vs # reacciones (tendencia lineal) | P3 — Exactitud |
| C3 | Severity vs # reacciones (boxplot) | Cross-dimension |
| C4 | logP vs # reacciones | Analítica |
| C5 | MW de fármacos por reacción (top 10) | Analítica |
| C6 | Fármacos compartidos entre top-12 reacciones | P2 — Consistencia |
| C7 | Anillos moleculares por cuartil de reacciones | Analítica |
| C8 | Cobertura del análisis molecular | P1 — Completitud |
| C9 | TPSA: neurológicos vs otros (barrera hematoencefálica) | Analítica |
| C10 | MW y logP promedio por reacción adversa | Analítica |


In [ ]:
print("─" * 70)
print("SECCIÓN C — Consultas Cross-Source (SIDER + PubChem)")
print("─" * 70)

# C1 — Completitud SMILES × # reacciones
print("\n  C1: Completitud SMILES × reacciones")
comp_data = df.groupby("drug_id").agg(
    n_reac=("adverse_reaction","nunique"),
    tiene_smiles=("smiles", lambda x: x.notna().any())
).reset_index()
fig, ax = plt.subplots(figsize=(9, 5))
for has, grp in comp_data.groupby("tiene_smiles"):
    ax.hist(grp["n_reac"], bins=40, alpha=0.7,
            color=("#4C72B0" if has else "#DD8452"),
            label=f"{'Con' if has else 'Sin'} SMILES")
ax.set_title("C1 — Completitud SMILES vs número de reacciones por fármaco")
ax.set_xlabel("# reacciones adversas únicas"); ax.set_ylabel("# fármacos"); ax.legend()
plt.tight_layout(); guardar(fig, GRAFICAS/"C1_completitud_smiles_vs_reacciones.png")

# C2 — MW vs # reacciones
print("  C2: MW vs # reacciones")
fig, ax = plt.subplots(figsize=(11, 6))
sc = ax.scatter(drug_stats["MW"], drug_stats["n_reac"], s=20, alpha=0.5,
                c=drug_stats["logP"], cmap="coolwarm")
plt.colorbar(sc, ax=ax, label="logP")
z = _np.polyfit(drug_stats["MW"], drug_stats["n_reac"], 1)
xline = _np.linspace(drug_stats["MW"].min(), drug_stats["MW"].max(), 200)
ax.plot(xline, _np.poly1d(z)(xline), "k--", lw=1.5, label="Tendencia lineal")
ax.set_title("C2 — Peso molecular vs reacciones adversas (SIDER + PubChem)")
ax.set_xlabel("Peso molecular (Da)"); ax.set_ylabel("# reacciones adversas únicas"); ax.legend(fontsize=8)
plt.tight_layout(); guardar(fig, GRAFICAS/"C2_MW_vs_reacciones.png")

# C3 — Severity × # reacciones
print("  C3: Severity × # reacciones")
ORDER_SEV_C = ["very_common","common","uncommon","rare","very_rare","postmarketing"]
sub_sev = df[df["severity"].isin(ORDER_SEV_C)]
fig, ax = plt.subplots(figsize=(11, 5))
sns.boxplot(data=sub_sev, x="severity", y="n_reac", ax=ax,
            order=ORDER_SEV_C, palette="RdYlGn_r")
ax.set_title("C3 — # reacciones del fármaco por categoría de severity")
ax.set_xlabel("Categoría de frecuencia de la reacción")
ax.set_ylabel("# reacciones adversas del fármaco")
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right")
plt.tight_layout(); guardar(fig, GRAFICAS/"C3_severity_vs_n_reacciones.png")

# C4 — logP vs # reacciones
print("  C4: logP vs # reacciones")
fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(drug_stats["logP"], drug_stats["n_reac"], s=18, alpha=0.45,
                c=drug_stats["MW"], cmap="plasma")
plt.colorbar(sc, ax=ax, label="MW (Da)")
z2 = _np.polyfit(drug_stats["logP"], drug_stats["n_reac"], 1)
xline2 = _np.linspace(drug_stats["logP"].min(), drug_stats["logP"].max(), 200)
ax.plot(xline2, _np.poly1d(z2)(xline2), "k--", lw=1.5, label="Tendencia")
ax.set_title("C4 — logP vs reacciones adversas (SIDER + PubChem)")
ax.set_xlabel("logP"); ax.set_ylabel("# reacciones adversas únicas"); ax.legend(fontsize=8)
plt.tight_layout(); guardar(fig, GRAFICAS/"C4_logP_vs_reacciones.png")

# C5 — Violin MW por reacción (top 10)
print("  C5: MW de fármacos por reacción (top 10)")
top10_c = df["adverse_reaction"].value_counts().head(10).index
sub_top = df[df["adverse_reaction"].isin(top10_c)].dropna(subset=["MW"])
fig, ax = plt.subplots(figsize=(13, 6))
sns.violinplot(data=sub_top, x="adverse_reaction", y="MW",
               order=top10_c, ax=ax, palette="Set2", inner="box")
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right", fontsize=8)
ax.set_title("C5 — Peso molecular de fármacos por reacción adversa (top 10)")
ax.set_xlabel(""); ax.set_ylabel("Peso Molecular (Da)")
plt.tight_layout(); guardar(fig, GRAFICAS/"C5_violin_MW_por_reaccion.png")

print("  C1–C5 completadas")


In [ ]:
# C6 — Heatmap fármacos compartidos entre top-12 reacciones
print("  C6: Fármacos compartidos entre top-12 reacciones")
top12 = df["adverse_reaction"].value_counts().head(12).index
sub12 = df[df["adverse_reaction"].isin(top12)][["drug_id","adverse_reaction"]]
piv12 = sub12.assign(v=1).pivot_table(
    index="drug_id", columns="adverse_reaction", values="v",
    aggfunc="max", fill_value=0)
co12 = piv12.T.dot(piv12)
_np.fill_diagonal(co12.values, 0)
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(co12, ax=ax, cmap="Blues", annot=True, fmt="d",
            linewidths=0.5, cbar_kws={"label":"# fármacos compartidos"})
ax.set_title("C6 — Fármacos compartidos entre top 12 reacciones adversas")
plt.xticks(rotation=35, ha="right", fontsize=7); plt.yticks(fontsize=7)
plt.tight_layout(); guardar(fig, GRAFICAS/"C6_heatmap_farmacos_compartidos.png")

# C7 — Anillos vs cuartil de reacciones
print("  C7: Anillos vs cuartil de reacciones")
drug_stats2 = drug_stats.merge(mol_df[["drug_id","n_rings"]], on="drug_id", how="left")
drug_stats2["cuartil_reac"] = pd.qcut(
    drug_stats2["n_reac"], q=4, labels=["Q1 (bajo)","Q2","Q3","Q4 (alto)"])
ring_mean = drug_stats2.groupby("cuartil_reac", observed=True)["n_rings"].mean()
fig, ax = plt.subplots(figsize=(9, 5))
bars_c7 = ax.bar(ring_mean.index, ring_mean.values,
                 color=sns.color_palette("Blues", 4), edgecolor="white")
ax.set_title("C7 — Promedio de anillos moleculares por cuartil de reacciones")
ax.set_xlabel("Cuartil de reacciones adversas"); ax.set_ylabel("Promedio de anillos")
for bar, val in zip(bars_c7, ring_mean.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
            f"{val:.2f}", ha="center", fontsize=9)
plt.tight_layout(); guardar(fig, GRAFICAS/"C7_anillos_vs_cuartil_reacciones.png")

# C8 — Cobertura del análisis molecular
print("  C8: Cobertura del análisis molecular")
total_farmacos    = df["drug_id"].nunique()
con_smiles_valido = df.dropna(subset=["MW"])["drug_id"].nunique()
sin_smiles_n      = total_farmacos - con_smiles_valido
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.pie([con_smiles_valido, sin_smiles_n],
        labels=["Con propiedades\nmoleculares","Sin propiedades\nmoleculares"],
        autopct="%1.1f%%", colors=["#4C72B0","#DD8452"], startangle=90)
ax1.set_title("C8a — Cobertura de propiedades moleculares")
cob_labels = ["Total","Con SMILES","Sin SMILES"]
cob_vals   = [total_farmacos, con_smiles_valido, sin_smiles_n]
ax2.barh(cob_labels, cob_vals, color=["#7fcdbb","#4C72B0","#DD8452"])
ax2.set_xlabel("# fármacos"); ax2.set_title("C8b — Impacto en cobertura del análisis")
for i, v in enumerate(cob_vals):
    ax2.text(v+2, i, str(v), va="center", fontsize=9)
plt.tight_layout(); guardar(fig, GRAFICAS/"C8_cobertura_analisis_molecular.png")

# C9 — TPSA: neurológicos vs otros
print("  C9: TPSA neurológicos vs otros")
neuro_kw = ["headache","dizziness","tremor","somnolence","insomnia",
            "seizure","neuropathy","confusion","anxiety","depression"]
df["es_neuro"] = df["adverse_reaction"].str.lower().apply(
    lambda x: any(k in str(x) for k in neuro_kw))
neuro_ids  = df[df["es_neuro"]]["drug_id"].unique()
drug_neuro = mol_df[mol_df["drug_id"].isin(neuro_ids)]["TPSA"]
drug_otros = mol_df[~mol_df["drug_id"].isin(neuro_ids)]["TPSA"]
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(drug_neuro, bins=40, alpha=0.6, color="#4C72B0",
        label=f"Neurológicos (n={len(drug_neuro)})", density=True)
ax.hist(drug_otros, bins=40, alpha=0.6, color="#DD8452",
        label=f"Otros (n={len(drug_otros)})", density=True)
ax.axvline(90, color="green", linestyle="--", label="TPSA=90 (BHE)")
ax.set_title("C9 — TPSA: fármacos con reacciones neurológicas vs otros")
ax.set_xlabel("TPSA (Å²)"); ax.set_ylabel("Densidad"); ax.legend(fontsize=8)
plt.tight_layout(); guardar(fig, GRAFICAS/"C9_tpsa_neuro_vs_otros.png")

# C10 — MW y logP por reacción adversa (top 20)
print("  C10: MW y logP por reacción adversa")
top20_reac = df["adverse_reaction"].value_counts().head(20).index
props_reac = (df[df["adverse_reaction"].isin(top20_reac)].dropna(subset=["MW","logP"])
              .groupby("adverse_reaction")
              .agg(n_drugs=("drug_id","nunique"),
                   MW_mean=("MW","mean"), logP_mean=("logP","mean"))
              .reset_index())
fig, ax = plt.subplots(figsize=(11, 7))
sc = ax.scatter(props_reac["MW_mean"], props_reac["logP_mean"],
                s=props_reac["n_drugs"]*3, alpha=0.7,
                c=range(len(props_reac)), cmap="tab20")
for _, row in props_reac.iterrows():
    ax.annotate(row["adverse_reaction"],
                (row["MW_mean"], row["logP_mean"]),
                textcoords="offset points", xytext=(5,3), fontsize=6.5)
ax.set_title("C10 — MW vs logP de fármacos por reacción adversa (tamaño=# fármacos)")
ax.set_xlabel("MW promedio (Da)"); ax.set_ylabel("logP promedio")
ax.text(0.02, 0.97, "Cada punto = reacción adversa\nTamaño = # fármacos",
        transform=ax.transAxes, fontsize=8, va="top",
        bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8))
plt.tight_layout(); guardar(fig, GRAFICAS/"C10_MW_logP_por_reaccion.png")

print("\n  Sección C completa — 10 gráficas generadas")


---
## 10. Sección D — Análisis Molecular: Fingerprints + Tanimoto + PCA + KMeans

**Pregunta central del proyecto:**
> ¿Los medicamentos que causan el mismo tipo de reacción adversa comparten subestructuras moleculares?

### Morgan Fingerprints (ECFP4)
Los **Extended Connectivity Fingerprints** (radio=2, 1024 bits) codifican la estructura molecular como un vector binario:
- Cada bit representa la presencia de una subestructura circular de radio r=2 alrededor de cada átomo
- El algoritmo es análogo al algoritmo de Morgan para numeración canónica de átomos
- Son la representación estándar en quimioinformática para comparación estructural

### Similitud Tanimoto (Coeficiente de Jaccard)
Para vectores binarios (fingerprints):
```
Tc(A, B) = |A ∩ B| / |A ∪ B| = bits_comunes / (bits_A + bits_B − bits_comunes)
```
- Rango: 0 (ningún bit común) a 1 (idénticas)
- Umbral operacional: Tc ≥ 0.4 para "similitud significativa" (Maggiora et al., 2014)
- Tc ≥ 0.85 = alta similitud estructural

### PCA sobre fingerprints
El espacio de fingerprints tiene 1,024 dimensiones. PCA reduce a 2D para visualización. La varianza explicada es baja (PC1≈6%, PC2≈4%) porque fingerprints binarios tienen alta dimensionalidad intrínseca.

### KMeans k=5
Los 5 clusters corresponden a arquetipos moleculares (familias farmacológicas: SSRI, AINE, antipsicóticos, antibióticos, esteroides). Silhouette score con k=2 es óptimo (0.40), pero k=5 ofrece mayor interpretabilidad farmacológica.


In [ ]:
print("─" * 70)
print("SECCIÓN D — Análisis Molecular")
print("─" * 70)
print("\n  Calculando Morgan fingerprints (radio=2, 1024 bits)...")

drug_fps = []
for _, r in drug_smiles.iterrows():
    try:
        mol = Chem.MolFromSmiles(r["smiles"])
        if mol:
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=1024)
            drug_fps.append({"drug_id": r["drug_id"], "name": r["name"], "fp": fp})
    except Exception:
        pass

print(f"  Fingerprints generados: {len(drug_fps):,} fármacos")

# Diccionario drug_id → fingerprint para consultas rápidas
fp_dict = {f["drug_id"]: f["fp"] for f in drug_fps}

# Matriz numpy para PCA/KMeans
fp_mat  = _np.array([list(f["fp"]) for f in drug_fps])
pca_ids = [f["drug_id"] for f in drug_fps]
print(f"  Matriz fingerprints: {fp_mat.shape}  (fármacos × bits)")


In [ ]:
print("\n  D1: Tanimoto similarity — fármacos con reacciones hepáticas")

all_reacs   = df["adverse_reaction"].unique()
hepa_reac   = [r for r in all_reacs if "hepat" in r.lower() or "liver" in r.lower()]
print(f"  Reacciones hepáticas encontradas: {len(hepa_reac)}")
if not hepa_reac:
    hepa_reac = df["adverse_reaction"].value_counts().head(1).index.tolist()

hepa_drug_ids = df[df["adverse_reaction"].isin(hepa_reac[:5])]["drug_id"].unique()
hepa_fps      = [f for f in drug_fps if f["drug_id"] in hepa_drug_ids][:20]
print(f"  Fármacos seleccionados: {len(hepa_fps)}")

n = len(hepa_fps)
if n >= 3:
    sim_matrix = _np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            sim_matrix[i, j] = DataStructs.TanimotoSimilarity(
                hepa_fps[i]["fp"], hepa_fps[j]["fp"])

    names_hepa = [f["name"][:18] for f in hepa_fps]
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(sim_matrix, xticklabels=names_hepa, yticklabels=names_hepa,
                ax=ax, cmap="RdYlGn", vmin=0, vmax=1,
                annot=(n<=12), fmt=".2f", linewidths=0.3,
                cbar_kws={"label":"Similitud Tanimoto"})
    ax.set_title(f"D1 — Similitud Tanimoto entre fármacos hepáticos\n"
                 f"(Morgan fp, radio=2, n={n})")
    plt.xticks(rotation=35, ha="right", fontsize=7); plt.yticks(fontsize=7)
    plt.tight_layout()
    guardar(fig, GRAFICAS/"D1_tanimoto_hepatotoxicidad.png")

    mean_sim = (sim_matrix.sum() - _np.trace(sim_matrix)) / (n*(n-1))
    print(f"  Similitud Tanimoto promedio intra-grupo (hepáticos): {mean_sim:.3f}")


In [ ]:
print("  D2: Distribución de Tanimoto por grupo de reacción (top 5)")

top5_reac  = df["adverse_reaction"].value_counts().head(5).index
tan_results = []
for reac in top5_reac:
    drug_ids_r  = df[df["adverse_reaction"]==reac]["drug_id"].unique()
    fps_reac    = [fp_dict[d] for d in drug_ids_r if d in fp_dict]
    if len(fps_reac) < 2: continue
    sample_fps = fps_reac[:30]
    sims = []
    for i in range(len(sample_fps)):
        for j in range(i+1, len(sample_fps)):
            sims.append(DataStructs.TanimotoSimilarity(sample_fps[i], sample_fps[j]))
    for s in sims:
        tan_results.append({"adverse_reaction": reac, "tanimoto": s})

tan_df = pd.DataFrame(tan_results)
fig, ax = plt.subplots(figsize=(11, 5))
sns.violinplot(data=tan_df, x="adverse_reaction", y="tanimoto",
               ax=ax, palette="Set1", inner="box")
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right", fontsize=8)
ax.set_title("D2 — Similitud Tanimoto intra-grupo por tipo de reacción (top 5)")
ax.set_xlabel(""); ax.set_ylabel("Similitud Tanimoto (Morgan fp)")
ax.axhline(0.4, color="red", linestyle="--", label="Umbral similaridad (0.4)")
ax.legend(fontsize=8)
plt.tight_layout()
guardar(fig, GRAFICAS/"D2_tanimoto_por_reaccion.png")

# Estadísticas por grupo
print("  Tanimoto promedio por grupo de reacción:")
for reac, grp in tan_df.groupby("adverse_reaction"):
    print(f"    {reac[:40]:<40} media={grp['tanimoto'].mean():.3f}  "
          f"pct≥0.4={100*(grp['tanimoto']>=0.4).mean():.1f}%")


In [ ]:
print("  D3: PCA de fingerprints moleculares (1024 dims → 2D)")

pca    = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(fp_mat)

pca_df = pd.DataFrame({
    "PC1"    : coords[:, 0],
    "PC2"    : coords[:, 1],
    "drug_id": pca_ids
}).merge(drug_stats[["drug_id","n_reac"]], on="drug_id", how="left")

print(f"  Varianza explicada: PC1={pca.explained_variance_ratio_[0]*100:.2f}%  "
      f"PC2={pca.explained_variance_ratio_[1]*100:.2f}%  "
      f"Total={sum(pca.explained_variance_ratio_)*100:.2f}%")
print("  (Nota: baja varianza explicada es esperada en fingerprints binarios 1024-bit)")

fig, ax = plt.subplots(figsize=(10, 7))
sc = ax.scatter(pca_df["PC1"], pca_df["PC2"],
                c=pca_df["n_reac"], cmap="plasma", s=15, alpha=0.6)
plt.colorbar(sc, ax=ax, label="# reacciones adversas")
ax.set_title(f"D3 — PCA de Morgan fingerprints moleculares\n"
             f"(PC1={pca.explained_variance_ratio_[0]*100:.1f}%  "
             f"PC2={pca.explained_variance_ratio_[1]*100:.1f}%)")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
plt.tight_layout()
guardar(fig, GRAFICAS/"D3_pca_fingerprints.png")


In [ ]:
print("  D4: KMeans clustering (k=5) sobre fingerprints")

from sklearn.metrics import silhouette_score

# Evaluar silhouette para k=2..6
print("  Silhouette score por k:")
for k in [2, 3, 4, 5, 6]:
    km_eval = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbl_eval = km_eval.fit_predict(fp_mat)
    sil = silhouette_score(fp_mat, lbl_eval, metric="jaccard",
                           sample_size=min(500, len(fp_mat)), random_state=42)
    print(f"    k={k}: silhouette={sil:.4f}")

# Modelo final k=5 (interpretabilidad farmacológica)
K = 5
km = KMeans(n_clusters=K, random_state=42, n_init=10)
clusters = km.fit_predict(fp_mat)
pca_df["cluster"] = clusters

fig, ax = plt.subplots(figsize=(10, 7))
for k in range(K):
    mask = pca_df["cluster"] == k
    ax.scatter(pca_df.loc[mask,"PC1"], pca_df.loc[mask,"PC2"],
               s=18, alpha=0.6, label=f"Cluster {k+1}")
ax.set_title(f"D4 — KMeans k={K} sobre fingerprints (espacio PCA)")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.legend(title="Cluster", fontsize=8)
plt.tight_layout()
guardar(fig, GRAFICAS/"D4_kmeans_fingerprints.png")

# Caracterización de clusters
print("\n  Caracterización de clusters:")
pca_merged = pca_df.merge(drug_stats[["drug_id","n_reac"]], on="drug_id", how="left",
                           suffixes=("","_ds"))
for k in range(K):
    mask_k = pca_df["cluster"] == k
    n_farmacos = mask_k.sum()
    drug_ids_k = pca_df[mask_k]["drug_id"].values
    n_reac_k   = df[df["drug_id"].isin(drug_ids_k)]["adverse_reaction"].nunique()
    top_reac_k = (df[df["drug_id"].isin(drug_ids_k)]["adverse_reaction"]
                  .value_counts().head(3).index.tolist())
    print(f"  Cluster {k+1}: {n_farmacos:>4} fármacos | "
          f"{n_reac_k:>5,} reacciones únicas | "
          f"Top reacciones: {', '.join(top_reac_k[:2])}")


---
## 11. Conclusiones

### 11.1 Resultados de Calidad de Datos (TDQM)

| Métrica | Antes | Después | Δ |
|---------|-------|---------|---|
| Filas totales | 152,759 | 152,759 | 0 |
| Severity conocida | 0.0% | 41.4% | +41.4 pp |
| SMILES válidos | 100.0% | 100.0% | = |
| Nombres limpios | ~100% | 100.0% | ≈ |
| Duplicados drug+reacción | 10,447 | 0 | –100% |

**Principales hallazgos de calidad:**
- **P1 (Completitud):** 93.6% de fármacos SIDER tienen SMILES en PubChem. El 6.4% sin SMILES son excluidos del análisis molecular.
- **P2 (Consistencia):** 53.9% de los fármacos tienen nombres SIDER e IUPAC con similitud Jaro-Winkler < 0.50 — justifica el uso de record linkage en lugar de joins por nombre.
- **P3 (Exactitud):** 100% de SMILES son parseables por RDKit (0 estructuras inválidas).
- **P4 (Unicidad):** 10,447 duplicados eliminados (6.4%) → 100% de unicidad.
- **P5 (Validez):** Severity imputada de 0% → 41.4% usando meddra_freq.tsv.

### 11.2 Respuesta a la Pregunta de Negocio P5

> **¿Comparten subestructuras moleculares los fármacos con la misma reacción adversa?**

**Evidencia a favor:**
- El heatmap de Tanimoto (D1) muestra **bloques de alta similitud** entre fármacos hepatotóxicos.
- El espacio PCA (D3) muestra **agrupamiento no aleatorio** — fármacos con perfiles similares se ubican cerca en el espacio molecular.
- El KMeans k=5 (D4) identifica **5 arquetipos moleculares** con perfiles de reacciones distinguibles.

**Limitaciones:**
- PC1+PC2 explican solo ~9.8% de la varianza (esperado para fingerprints binarios 1024-bit).
- La relación estructura→reacción no es determinista; el metabolismo individual y las interacciones también influyen.
- Análisis futuro: GNN (Graph Neural Networks) sobre grafos moleculares para mayor precisión predictiva.

### 11.3 Archivos Generados

| Archivo | Ubicación | Descripción |
|---------|-----------|-------------|
| `e_drugDB.csv` | `Bases_datos/clean/` | Modelo canónico completo |
| `e_drugDB_smiles.csv` | `Bases_datos/clean/` | Solo registros con SMILES |
| `e_drugDB_clean.csv` | `Bases_datos/clean/` | Dataset limpio final |
| `linkage_report.csv` | `Bases_datos/clean/` | Reporte de record linkage |
| `perfil_antes_limpieza.html` | `Bases_datos/clean/perfiles/` | ydata-profiling antes |
| `perfil_despues_limpieza.html` | `Bases_datos/clean/perfiles/` | ydata-profiling después |
| Gráficas 01–13 (PNG) | `Bases_datos/clean/perfiles/` | Perfilado antes/después |
| Gráficas A1–A5, B1–B5, C1–C10, D1–D4 (PNG) | `Bases_datos/clean/graficas/` | Análisis SIDER · PubChem · Cross-source · Molecular |
